# 03 — Visualizaciones del programa de gobierno

Genera las gráficas del análisis:
- Distribución de tipos de enunciado
- Propuestas por área temática
- Verificabilidad de cifras
- WordCloud del corpus completo
- Comparación resumen ejecutivo vs. pilares (cobertura temática)
- Cifras por área temática


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

PROCESSED = Path('../data/processed')
OUTPUTS = Path('../outputs')
VIZ = Path('../visualizations')
VIZ.mkdir(exist_ok=True)

PALETTE = {
    'primario':   '#1B3A6B',
    'secundario': '#C8102E',
    'acento':     '#F5A623',
    'gris':       '#6B7280',
    'claro':      '#F3F4F6',
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
})

## 1. Distribución de tipos de enunciado

In [ ]:
# Datos extraídos del análisis manual del corpus
tipos = {
    'Narrativa': 31,
    'Propuesta': 27,
    'Diagnóstico': 18,
    'Ataque político': 10,
    'Marco ideológico': 8,
    'Cifra': 4,
    'Promesa vaga': 2,
}

colores = [
    PALETTE['primario'], PALETTE['acento'], PALETTE['secundario'],
    '#9B2335', '#2E5F8A', '#4CAF50', PALETTE['gris']
]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(list(tipos.keys()), list(tipos.values()), color=colores, height=0.6)

for bar, val in zip(bars, tipos.values()):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height()/2,
            f'{val}%', va='center', fontsize=10, color=PALETTE['gris'])

ax.set_xlabel('Porcentaje del corpus (%)', fontsize=11)
ax.set_title('Distribución de tipos de enunciado\nCorpus: Programa Colombia, Patria Milagro',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, 40)
ax.tick_params(axis='y', labelsize=11)
fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '01_tipos_enunciado.png', bbox_inches='tight')
plt.show()

## 2. Propuestas por área temática

In [ ]:
areas = {
    'Seguridad y control territorial': 24,
    'Anticorrupción': 18,
    'Economía y modelo de desarrollo': 14,
    'Campo y agro': 10,
    'Salud': 9,
    'Energía y minería': 8,
    'Educación': 6,
    'Mujeres y cuidado': 5,
    'Democracia e instituciones': 4,
    'Cultura': 2,
}

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(areas))
valores = list(areas.values())
etiquetas = list(areas.keys())

colores_areas = [PALETTE['primario'] if v >= 10 else PALETTE['secundario'] if v >= 6 else PALETTE['gris']
                 for v in valores]

bars = ax.barh(y_pos, valores, color=colores_areas, height=0.65)
ax.set_yticks(y_pos)
ax.set_yticklabels(etiquetas, fontsize=10)
ax.set_xlabel('Número de propuestas identificadas', fontsize=11)
ax.set_title('Propuestas por área temática\nCorpus: Programa Colombia, Patria Milagro',
             fontsize=13, fontweight='bold', pad=15)

for bar, val in zip(bars, valores):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, color=PALETTE['gris'])

patch_alto = mpatches.Patch(color=PALETTE['primario'], label='Alta densidad (≥10)')
patch_medio = mpatches.Patch(color=PALETTE['secundario'], label='Media densidad (6–9)')
patch_bajo = mpatches.Patch(color=PALETTE['gris'], label='Baja densidad (<6)')
ax.legend(handles=[patch_alto, patch_medio, patch_bajo], fontsize=9, loc='lower right')

fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '02_propuestas_por_area.png', bbox_inches='tight')
plt.show()

## 3. Verificabilidad de cifras

In [ ]:
verificabilidad = {'Verificable': 12, 'Parcialmente\nverificable': 13, 'Sin fuente\nexplícita': 5}

colores_v = ['#2E7D32', PALETTE['acento'], PALETTE['secundario']]
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    verificabilidad.values(),
    labels=verificabilidad.keys(),
    colors=colores_v,
    autopct='%1.0f%%',
    startangle=140,
    pctdistance=0.75,
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'},
)
for t in autotexts:
    t.set_fontsize(13)
    t.set_fontweight('bold')
    t.set_color('white')

ax.set_title('Verificabilidad de las 30 cifras extraídas\ndel corpus programático',
             fontsize=13, fontweight='bold', pad=20)
fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '03_verificabilidad_cifras.png', bbox_inches='tight')
plt.show()

## 4. Cobertura temática: resumen ejecutivo vs. pilares

In [ ]:
temas = [
    'Seguridad', 'Anticorrupción', 'Economía', 'Salud', 'Campo',
    'Energía', 'Educación', 'Mujeres', 'Democracia', 'Cultura',
    'Medio\nAmbiente', 'Marco\nideológico'
]
resumen = [2, 2, 3, 2, 2, 2, 2, 2, 1, 1, 1, 0]
pilares  = [5, 5, 4, 4, 4, 4, 3, 3, 3, 2, 2, 5]

x = np.arange(len(temas))
w = 0.38

fig, ax = plt.subplots(figsize=(13, 6))
b1 = ax.bar(x - w/2, resumen, w, label='Resumen ejecutivo (3 págs.)',
            color=PALETTE['acento'], alpha=0.9)
b2 = ax.bar(x + w/2, pilares, w, label='Pilares completos',
            color=PALETTE['primario'], alpha=0.9)

ax.set_xticks(x)
ax.set_xticklabels(temas, fontsize=9)
ax.set_ylabel('Profundidad de cobertura (0–5)', fontsize=11)
ax.set_yticks(range(6))
ax.set_yticklabels(['Ausente', 'Mencionado', 'Resumido', 'Desarrollado', 'Extenso', 'Central'], fontsize=9)
ax.set_title('Cobertura temática: Resumen ejecutivo vs. Pilares completos\nColombia, Patria Milagro — Abelardo de la Espriella 2026',
             fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.set_ylim(0, 6)

fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '04_cobertura_resumen_vs_pilares.png', bbox_inches='tight')
plt.show()

## 5. Cifras por área temática

In [ ]:
df_cifras = pd.read_csv('../outputs/cifras_extraidas.csv')
conteo = df_cifras['area_tematica'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(conteo.index, conteo.values,
              color=[PALETTE['primario'] if v >= 5 else PALETTE['secundario'] if v >= 3 else PALETTE['gris']
                     for v in conteo.values])

for bar, val in zip(bars, conteo.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            str(val), ha='center', fontsize=10)

ax.set_xlabel('Área temática', fontsize=11)
ax.set_ylabel('N° de cifras extraídas', fontsize=11)
ax.set_title('Cifras identificadas por área temática\n(de 30 cifras totales extraídas del corpus)',
             fontsize=13, fontweight='bold', pad=15)
plt.xticks(rotation=30, ha='right', fontsize=9)
fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '05_cifras_por_area.png', bbox_inches='tight')
plt.show()

## 6. WordCloud del corpus

In [ ]:
from wordcloud import WordCloud

# Términos más frecuentes extraídos del corpus (frecuencias estimadas del análisis)
frecuencias = {
    'patria': 180, 'Colombia': 160, 'nación': 120, 'seguridad': 95,
    'corrupción': 88, 'Estado': 85, 'crimen': 80, 'gobierno': 78,
    'República': 72, 'democracia': 68, 'Constitución': 65, 'libertad': 60,
    'fuerza pública': 55, 'territorios': 52, 'ciudadano': 50, 'familia': 48,
    'restauración': 45, 'economía': 44, 'narcotráfico': 42, 'extorsión': 40,
    'verdad': 38, 'salud': 36, 'campo': 35, 'autoridad': 34, 'principios': 32,
    'dignidad': 30, 'bien común': 28, 'Ecopetrol': 26, 'fiscal': 25,
    'minería': 24, 'educación': 23, 'mujeres': 22, 'energía': 21,
    'anticorrupción': 20, 'los nunca': 18, 'blockchain': 15,
}

wc = WordCloud(
    width=1200, height=600,
    background_color='white',
    colormap='RdYlBu_r',
    max_words=60,
    prefer_horizontal=0.85,
    min_font_size=10,
).generate_from_frequencies(frecuencias)

fig, ax = plt.subplots(figsize=(14, 7))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Términos más frecuentes — Corpus Colombia, Patria Milagro',
             fontsize=14, fontweight='bold', pad=15)
fig.text(0.99, 0.01, 'github.com/DavidMume/patria-milagro-analysis',
         ha='right', fontsize=7, color=PALETTE['gris'])
plt.tight_layout()
plt.savefig(VIZ / '06_wordcloud_corpus.png', bbox_inches='tight')
plt.show()